In [1]:
%pip install pandas
%pip install wordninja
%pip install deep-translator
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import wordninja
import os
import string
from langdetect import detect
from deep_translator import GoogleTranslator


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
#load dataset
df = pd.read_csv('cricket_data.csv')
print(df.head())


                                   title  \
0                                Cricket   
1  Board of Control for Cricket in India   
2            India national cricket team   
3                        Women's cricket   
4                           Test cricket   

                                             summary  \
0                                                NaN   
1  \nTheBoard of Control for Cricket in India, al...   
2  \nTheIndia men's national cricket team, repres...   
3  \nWomen's cricketis theteam sportofcricketwhen...   
4  \nTest Cricketis aformatof the sport ofcricket...   

                                             content  \
0                                                NaN   
1  \nPrabhtej Bhatia\n(Treasurer)[7]\nAjit Agarka...   
2  \nTestkit\nODIkit\nT20Ikit\nTheIndia men's nat...   
3  \nMen's format\nWomen's format\nFirst-class cr...   
4  \nBall,\nBat,\nStumps,\nCricket helmet,\nThigh...   

                                               links  \
0    

In [3]:
# checking for missing values
missing_values = df.isnull()
for column in missing_values.columns.values.tolist():
    print (missing_values[column].value_counts())
    print("")

title
False    10000
Name: count, dtype: int64

summary
False    8895
True     1105
Name: count, dtype: int64

content
False    8908
True     1092
Name: count, dtype: int64

links
False    10000
Name: count, dtype: int64

url
False    10000
Name: count, dtype: int64



In [4]:
#display the duplicate rows
duplicate_rows_df = df[df.duplicated()]
print("number of duplicate rows: ", duplicate_rows_df.shape)

number of duplicate rows:  (1957, 5)


In [5]:
#dropping duplicates
df = df.drop_duplicates()
print("number of duplicate rows: ", df.duplicated())

number of duplicate rows:  0       False
1       False
2       False
3       False
4       False
        ...  
9994    False
9996    False
9997    False
9998    False
9999    False
Length: 8043, dtype: bool


In [6]:
# Drop rows where both columns 'content' and 'summary' are empty
df1 = df.copy()
df_cleaned = df1.dropna(subset=['content', 'summary'], how='all')

In [7]:
#checking for missing values after removing duplicates
missing_valuesdf1 = df_cleaned.isnull()
for column in missing_valuesdf1.columns.values.tolist():
    print (missing_valuesdf1[column].value_counts())
    print("")

title
False    7163
Name: count, dtype: int64

summary
False    7154
True        9
Name: count, dtype: int64

content
False    7163
Name: count, dtype: int64

links
False    7163
Name: count, dtype: int64

url
False    7163
Name: count, dtype: int64



In [8]:
# having only 9 rows with missing we can replace them with Not Available

df1 = df.copy()

df_cleaned = df1.dropna(subset=['summary'], how='all').copy()

# Function to extract first 25 words
def get_summary(text):
    words = text.split()
    return ' '.join(words[:25]) if len(words) > 25 else text

df_cleaned.loc[df_cleaned['summary'].isna(), 'summary'] = df_cleaned.loc[df_cleaned['summary'].isna(), 'content'].apply(get_summary)


In [9]:
# checking if there are any missing values left
missing_valuesdf1 = df_cleaned.isnull()
for column in missing_valuesdf1.columns.values.tolist():
    print (missing_valuesdf1[column].value_counts())
    print("")

title
False    7154
Name: count, dtype: int64

summary
False    7154
Name: count, dtype: int64

content
False    7154
Name: count, dtype: int64

links
False    7154
Name: count, dtype: int64

url
False    7154
Name: count, dtype: int64



In [10]:
#removing \n from the content table and summary
df_cleaned["content"] = df_cleaned["content"].str.replace("\n", " ", regex=True)
df_cleaned["summary"] = df_cleaned["summary"].str.replace("\n", " ", regex=True)

In [11]:
df_cleaned.head()


,title,summary,content,links,url
1,Board of Control for Cricket in India,"TheBoard of Control for Cricket in India, als...",Prabhtej Bhatia (Treasurer)[7] Ajit Agarkar (...,['/wiki/Delhi_%26_District_Cricket_Association...,https://en.wikipedia.org/wiki/Board_of_Control...
2,India national cricket team,"TheIndia men's national cricket team, represe...",Testkit ODIkit T20Ikit TheIndia men's nationa...,"['/wiki/Ambuja_Cements', '/wiki/South_African_...",https://en.wikipedia.org/wiki/India_national_c...
3,Women's cricket,Women's cricketis theteam sportofcricketwhen ...,Men's format Women's format First-class crick...,"['/wiki/Peru_women%27s_national_cricket_team',...",https://en.wikipedia.org/wiki/Women's_cricket
4,Test cricket,"Test Cricketis aformatof the sport ofcricket,...","Ball, Bat, Stumps, Cricket helmet, Thigh guar...","['/wiki/Swing_bowling#Reverse_swing', '/wiki/F...",https://en.wikipedia.org/wiki/Test_cricket
5,Glossary of cricket terms,,This is a generalglossaryof the terminology ...,"['/wiki/Boundary_rope', '/wiki/Jockstrap', '/w...",https://en.wikipedia.org/wiki/Glossary_of_cric...


In [12]:
# Apply word splitting
df_cleaned['summary'] =df_cleaned['summary'].apply(lambda x: " ".join(wordninja.split(x)))
df_cleaned['content'] =df_cleaned['content'].apply(lambda x: " ".join(wordninja.split(x)))

In [ ]:
# Initialize GoogleTranslator *once* outside the function
translator = GoogleTranslator(source='auto', target='en')

# Function to detect language
def detect_language(text):
    if not isinstance(text, str): # Make sure it is a string
        return "unknown"  # Handle non-string values
    try:
        return detect(text)
    except Exception as e:
        print(f"Language detection error: {e}")
        return "unknown"  # Handle errors

# Function to translate text
def translate_text(text):
    """Translates text to English, handling potential errors and NaN values."""
    if not isinstance(text, str) or pd.isna(text): # Add pd.isna check
        return None  # Or some other appropriate placeholder like "" or "[Translation Unavailable]"
    try:
        return translator.translate(text)  # Use the pre-initialized translator
    except Exception as e:
        print(f"Translation error: {e}")
        return None  # Handle translation errors

def translate_dataframe(df):
    """Translates non-English content in a DataFrame to English."""

    # Ensure 'content' column exists
    if 'content' not in df.columns:
        print("Error: 'content' column not found in DataFrame.")
        return df  # Or raise an exception

    # Convert 'content' column to string type and fill NaN values with empty string
    df['content'] = df['content'].astype(str).fillna('')  #This will be converted to string to avoid float
    # 1. Detect Language
    df['Language'] = df['content'].apply(detect_language)

    # 2. Filter only non-English rows
    non_english_df = df[df['Language'] != 'en'].copy()

    # 3. Translate Content, check the non_english_df is empty before applying the translation
    if not non_english_df.empty:
        non_english_df['EnglishText'] = non_english_df['content'].apply(translate_text)
    else:
        non_english_df['EnglishText'] = None

    # 4. Merge translated texts back into original DataFrame
    # Create EnglishText column if it doesn't exist (to handle cases where all text is already English)
    if 'EnglishText' not in df.columns:
         df['EnglishText'] = None

    # Update the EnglishText column in df with the translated values from non_english_df
    df.loc[non_english_df.index, 'EnglishText'] = non_english_df['EnglishText']

    return df


df_translated = translate_dataframe(df)
print(df_translated)

In [ ]:
df_cleaned['links'] = df['links']

df_cleaned.head()

,title,summary,content,links,url
1,Board of Control for Cricket in India,The Board of Control for Cricket in India also...,Pra bh te j Bhat i a Treasurer 7 A j it Agar k...,['/wiki/Delhi_%26_District_Cricket_Association...,https://en.wikipedia.org/wiki/Board_of_Control...
2,India national cricket team,The India men's national cricket team represen...,Test kit ODI kit T 20 Ik it The India men's na...,"['/wiki/Ambuja_Cements', '/wiki/South_African_...",https://en.wikipedia.org/wiki/India_national_c...
3,Women's cricket,Women's cricket is the team sport of cricket w...,Men's format Women's format First class cricke...,"['/wiki/Peru_women%27s_national_cricket_team',...",https://en.wikipedia.org/wiki/Women's_cricket
4,Test cricket,Test Cricket is a format of the sport of crick...,Ball Bat Stumps Cricket helmet Thigh guard Bat...,"['/wiki/Swing_bowling#Reverse_swing', '/wiki/F...",https://en.wikipedia.org/wiki/Test_cricket
5,Glossary of cricket terms,,This is a general glossary of the terminology ...,"['/wiki/Boundary_rope', '/wiki/Jockstrap', '/w...",https://en.wikipedia.org/wiki/Glossary_of_cric...


In [ ]:
# removing links from the summary, content and title columns


def remove_url(text):
    return re.sub(r'https?://\S+|www\.\S+', '', text)

#This function removes punctuations
def remove_punct(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df_cleaned['content'] = df_cleaned['content'].apply(lambda x: remove_url(x))
df_cleaned['summary'] = df_cleaned['summary'].apply(lambda x: remove_url(x))
df_cleaned['title'] = df_cleaned['title'].apply(lambda x: remove_url(x))

df_cleaned = df_cleaned.drop(columns=['links'])

In [ ]:
df_cleaned.tail(10)

,title,summary,content,url
9984,Wade Seccombe,Wade Anthony Sec combe born 30 October 1971 in...,Wade Anthony Sec combe born 30 October 1971 in...,https://en.wikipedia.org/wiki/Wade_Seccombe
9985,Sachith Pathirana,S achi th Shan aka Path iran a born 21 March 1...,Sri Lanka 2015 2017 S achi th Shan aka Path ir...,https://en.wikipedia.org/wiki/Sachith_Pathirana
9987,Renuka Singh Thakur,Ren uk a Singh Thakur born 2 January 1996 is a...,India Ren uk a Singh Thakur born 2 January 199...,https://en.wikipedia.org/wiki/Renuka_Singh_Thakur
9989,Subroto Banerjee,Sub ro to Tara Banerjee pronunciation born 13 ...,India Sub ro to Tara Banerjee pronunciation bo...,https://en.wikipedia.org/wiki/Subroto_Banerjee
9991,1954–55 Sheffield Shield season,The 1954 55 Sheffield Shield season was the 53...,The 1954 55 Sheffield Shield season was the 53...,https://en.wikipedia.org/wiki/1954–55_Sheffiel...
9993,Don Bennett (cricketer),Donald Bennett 18 December 1933 12 June 2014 w...,Donald Bennett 18 December 1933 12 June 2014 w...,https://en.wikipedia.org/wiki/Don_Bennett_(cri...
9996,1912 Liga Peruana de Football,The 1912 Primera Divisi n was the first season...,The 1912 Primera Divisi n was the first season...,https://en.wikipedia.org/wiki/1912_Liga_Peruan...
9997,Gary Ballance,Gary Simon Ball ance born 22 November 1989 1 i...,England 2013 2017 Zimbabwe 2023 2023 Gary Simo...,https://en.wikipedia.org/wiki/Gary_Ballance
9998,Wills World Series,The 1994 95 Wills World Series named after spo...,The 1994 95 Wills World Series named after spo...,https://en.wikipedia.org/wiki/Wills_World_Series
9999,1906–07 Sheffield Shield season,The 1906 07 Sheffield Shield season was the 15...,The 1906 07 Sheffield Shield season was the 15...,https://en.wikipedia.org/wiki/1906–07_Sheffiel...


In [ ]:
#resetting the index
df_cleaned = df_cleaned.reset_index(drop=True)  # drop=True removes old index

In [ ]:
df_cleaned.tail(20)

,title,summary,content,url,Language
7134,Tinashe Panyangara,Tina she P any angara born 21 October 1985 in ...,Zimbabwe Tina she P any angara born 21 October...,https://en.wikipedia.org/wiki/Tinashe_Panyangara,en
7135,Olivia Porter (cricketer),Olivia Kate Porter born 14 November 2001 is an...,Olivia Kate Porter born 14 November 2001 is an...,https://en.wikipedia.org/wiki/Olivia_Porter_(c...,NaN
7136,Peter Randall Johnson,Peter Randall Johnson 5 August 1880 1 July 195...,George Randall Johnson father Richard Cubitt J...,https://en.wikipedia.org/wiki/Peter_Randall_Jo...,en
7137,Joe B. Mauldin,Joseph Benson Maul d in Jr July 8 1940 Februar...,Joseph Benson Maul d in Jr July 8 1940 Februar...,https://en.wikipedia.org/wiki/Joe_B._Mauldin,en
7138,Andrew Mansale,Andrew Man sale born 5 August 1988 is a Vanuat...,Vanuatu Andrew Man sale born 5 August 1988 is ...,https://en.wikipedia.org/wiki/Andrew_Mansale,en
7139,Iain Brunnschweiler,Iain Brunn sch weil er born 10 December 1979 i...,Iain Brunn sch weil er born 10 December 1979 i...,https://en.wikipedia.org/wiki/Iain_Brunnschweiler,en
7140,Dayle Hadlee,Day le Robert Had lee born 6 January 1948 is a...,New Zealand 1969 1978 Day le Robert Had lee bo...,https://en.wikipedia.org/wiki/Dayle_Hadlee,en
7141,Karachi Dolphins,The Karachi Dolphins were a limited over s cri...,The Karachi Dolphins were a limited over s cri...,https://en.wikipedia.org/wiki/Karachi_Dolphins,en
7142,Zaka Ashraf,Chaudhry Muhammad Zak a Ashraf Punjabi Urdu bo...,Chaudhry Muhammad Ashraf father Businessman Ad...,https://en.wikipedia.org/wiki/Zaka_Ashraf,en
7143,Rockie D'Mello,Rock ie D ' Mello born 26 October 1961 is an I...,Rock ie D ' Mello born 26 October 1961 is an I...,https://en.wikipedia.org/wiki/Rockie_D'Mello,en
